# Block 2 — Assignment 2

**Group G — Team members**

- Marie Lacroix
- Chloé Baruselli
- Bertille Delloye
- Enéa Drezet--Marçot


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyiast

## Question 1 — Types of adsorption calculations available in AiiDAlab

AiiDAlab offers three main types of calculations used in this assignment:

1. **Pore analysis (Zeo++)**  
   This characterizes the geometry of the porous material and provides quantities such as density, accessible surface area (ASA), accessible probe-occupiable volume (POAV), porosity, channels and pockets.

2. **Henry coefficient / Widom insertion**  
   This probes adsorption in the **low-pressure / low-loading limit**. A larger Henry coefficient means a stronger initial affinity of the material for the gas.

3. **Adsorption isotherm / GCMC**  
   Grand Canonical Monte Carlo (GCMC) simulations calculate the amount of gas adsorbed as a function of pressure at a fixed temperature.

Together, these calculations answer three complementary questions: **Is the pore space accessible? How strongly does the gas interact with the material at low loading? How much gas is adsorbed as pressure increases?**

## Question 2 — Pore analysis of IRMOF-1

The pore analysis was performed using a probe radius of

$$
r_{\mathrm{probe}} = 1.525\ \text{\AA}.
$$

The results are:

| Property | Value |
|---|---:|
| Density | $0.576983\ \mathrm{g\,cm⁻³}$ |
| Accessible surface area (ASA) | $3964.65\ \mathrm{\AA^2}$ |
| Accessible probe-occupiable volume (POAV) | $13737.4\ \mathrm{\AA^3}$ |
| POAV volume fraction / accessible porosity | $0.77498$ |

Therefore,

$$
\phi = 0.77498 \approx 77.5\%.
$$

The calculation also identified **one accessible channel and no inaccessible pockets**. This is consistent with a highly porous material in which a large fraction of the unit-cell volume is accessible to the probe.

The ASA describes the internal surface that can be reached by the probe, while the POAV describes the volume in which the probe can physically be accommodated. These quantities depend on the probe radius because a smaller probe can access regions that may be inaccessible to a larger one.

## Question 3 — Henry coefficients at 300 K

The calculated Henry coefficients are

$$
K_{H,\mathrm{CO_2}} = 4.95112 \times 10^{-6}\ \mathrm{mol\,kg^{-1}\,Pa^{-1}}
$$

and

$$
K_{H,\mathrm{CH_4}} = 1.12113 \times 10^{-6}\ \mathrm{mol\,kg^{-1}\,Pa^{-1}}.
$$

At low pressure,

$$
q_i \approx K_{H,i} P_i.
$$

The Henry coefficient of CO₂ is about **4.4 times larger** than that of CH₄. This means that, in the low-pressure limit, IRMOF-1 has a substantially stronger adsorption affinity for CO₂ than for CH₄.

This suggests preferential CO₂ adsorption from a CO₂/CH₄ mixture. However, Henry coefficients only describe the low-loading limit, so finite-pressure behaviour must be examined using adsorption isotherms and, for the actual mixture, IAST.

## Question 4 — Pure-component adsorption isotherms at 300 K

The pure CO₂ and CH₄ adsorption isotherms were obtained from the AiiDAlab simulations over the requested pressure range.

The same pressure points are used for both gases. The loadings are reported in `mol/kg`, which is numerically equivalent to `mmol/g`:

$$
1\ \mathrm{mol\,kg^{-1}} = 1\ \mathrm{mmol\,g^{-1}}.
$$

In [ ]:
# Pure-component adsorption data obtained from AiiDAlab at 300 K

pressure = np.array([
    0.2, 0.8, 1.4, 2.0, 2.6, 3.2, 3.8, 4.4, 5.0,
    5.6, 6.2, 6.8, 7.4, 8.0, 8.6, 9.2, 9.8, 10.2
])

loading_CO2 = np.array([
    0.09649599694768, 0.3948153845504, 0.69862192675748,
    1.02791613976, 1.4028933499715, 1.6763744341876,
    2.166452017609, 2.50301914559, 2.9011138355549,
    3.2198882965348, 3.655126802283, 3.9100359786074,
    4.4598553636384, 5.0184412086882, 5.4556602850332,
    5.9664851571657, 6.7251411008668, 7.0717084869433
])

loading_CH4 = np.array([
    0.02162393471304, 0.08678795418612, 0.15549102603716,
    0.22643441544856, 0.29000748477008, 0.35429485824128,
    0.42897211025328, 0.48952562112388, 0.56426780987676,
    0.6243667635612, 0.70274540980336, 0.76359113600792,
    0.81742369419744, 0.88739303249564, 0.94245938876188,
    1.0342474719958, 1.1067168748178, 1.1316525833158
])

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(pressure, loading_CO2, marker="o", label="CO₂")
plt.plot(pressure, loading_CH4, marker="o", label="CH₄")

plt.xlabel("Pressure (bar)")
plt.ylabel("Loading (mmol/g)")
plt.title("Pure-component adsorption isotherms at 300 K")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Comparison and interpretation

For both gases, the adsorbed amount increases with pressure. However, the CO₂ loading is much larger than the CH₄ loading throughout the simulated pressure range.

This is consistent with Question 3, where CO₂ also had the larger Henry coefficient. IRMOF-1 therefore shows a stronger affinity for CO₂ than for CH₄.

The pure-component isotherms do **not** directly describe competitive adsorption in a CO₂/CH₄ mixture. This is why the next step is to use these two pure isotherms as inputs for IAST.

## Question 5 — Binary-mixture isotherms using IAST

Ideal Adsorbed Solution Theory (IAST) predicts the adsorption of a gas mixture from the pure-component adsorption isotherms of each gas. The pure CO₂ and CH₄ isotherms obtained from the GCMC simulations at 300 K are then used for the IAST calculation of the CO₂/CH₄ mixture in IRMOF-1.

The assignment specifically requires the linear interpolation method, so the pure-component data are represented using `pyiast.InterpolatorIsotherm` rather than fitted to an analytical model.

For a mixture at total pressure \(P\), the partial pressure of each component is

$$ p_i=y_iP. $$

For the biogas composition considered here,

$$
y_{\mathrm{CO_2}} = 0.40,\qquad
y_{\mathrm{CH_4}} = 0.60
$$


In [ ]:
co2_df = pd.DataFrame({
    "pressure": pressure,
    "loading": loading_CO2
})

ch4_df = pd.DataFrame({
    "pressure": pressure,
    "loading": loading_CH4
})



### Linear interpolation of the pure-component isotherms

The GCMC simulations provide adsorption loadings only at specific pressure values. Linear interpolation is used to estimate the missing values between two neighbouring simulation points by assuming that the isotherm varies linearly between them.

The `InterpolatorIsotherm` function applies this procedure to the CO₂ and CH₄ pure-component isotherms. These continuous representations are then used in the IAST calculation to determine the adsorption of both gases in the mixture.


In [ ]:
co2_isotherm = pyiast.InterpolatorIsotherm(
    co2_df,
    loading_key="loading",
    pressure_key="pressure"
)

ch4_isotherm = pyiast.InterpolatorIsotherm(
    ch4_df,
    loading_key="loading",
    pressure_key="pressure"
)


In [ ]:
y = np.array([0.40, 0.60])
# index 0 = CO2
# index 1 = CH4

In [ ]:
pressures_mix = np.arange(0.1, 4.0, 0.2)

q_CO2_mix = []
q_CH4_mix = []

for P in pressures_mix:
    q = pyiast.iast(
        P * y,
        [co2_isotherm, ch4_isotherm]
    )

    q_CO2_mix.append(q[0])
    q_CH4_mix.append(q[1])

q_CO2_mix = np.array(q_CO2_mix)
q_CH4_mix = np.array(q_CH4_mix)

In [ ]:
mixture_df = pd.DataFrame({
    "Pressure (bar)": pressures_mix,
    "CO2 loading (mmol/g)": q_CO2_mix,
    "CH4 loading (mmol/g)": q_CH4_mix
})

mixture_df

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    pressures_mix,
    q_CO2_mix,
    marker="o",
    label="CO₂"
)

plt.plot(
    pressures_mix,
    q_CH4_mix,
    marker="o",
    label="CH₄"
)

plt.xlabel("Total pressure (bar)")
plt.ylabel("Binary loading (mmol/g)")
plt.title("IAST binary adsorption isotherms at 300 K")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

The binary loadings differ from the pure-component loadings because CO₂ and CH₄ are now present simultaneously and compete for adsorption within the same pore space. In addition, each component is exposed to its own partial pressure rather than to the full total pressure.

The IAST results show a larger adsorbed amount of CO₂ than CH₄. This indicates preferential adsorption of CO₂: the composition of the adsorbed phase is enriched in CO₂ relative to the gas phase.

This result is consistent with Question 3, where the Henry coefficient of CO₂ was significantly larger than that of CH₄. The magnitude of this preference will be quantified by the selectivity in Question 6.

## Question 6 — CO₂/CH₄ selectivity at 300 K

The selectivity describes the preference of the material for one gas over another during competitive adsorption. For CO₂ over CH₄, it is defined as

$$
S_{\mathrm{CO_2/CH_4}}
=
\frac{q_{\mathrm{CO_2}}}{q_{\mathrm{CH_4}}}
\frac{y_{\mathrm{CH_4}}}{y_{\mathrm{CO_2}}}
$$

where \(q\) is the amount of each gas adsorbed in the CO₂/CH₄ mixture, as predicted by IAST, while \(y\) represents the fraction of each gas in the gas phase.

The selectivity compares the relative amount of CO₂ and CH₄ adsorbed with their relative amounts in the original gas mixture. A value larger than 1 means that CO₂ is preferentially adsorbed over CH₄.


In [ ]:
selectivity_pressures = [0.1, 1.0, 2.0, 3.0]

results = []

for P in selectivity_pressures:

    # Binary loadings from IAST
    q = pyiast.iast(
        P * y,
        [co2_isotherm, ch4_isotherm]
    )

    q_CO2 = q[0]
    q_CH4 = q[1]

    # CO2/CH4 selectivity
    S = (q_CO2 / q_CH4) * (y[1] / y[0])

    results.append([P, q_CO2, q_CH4, S])

selectivity_df = pd.DataFrame(
    results,
    columns=[
        "Pressure (bar)",
        "CO2 loading (mmol/g)",
        "CH4 loading (mmol/g)",
        "CO2/CH4 selectivity"
    ]
)

selectivity_df

### Interpretation

The CO₂/CH₄ selectivity remains approximately constant over the studied pressure range and is clearly greater than 1.

This indicates that IRMOF-1 preferentially adsorbs CO₂ over CH₄ throughout this pressure range, even though CH₄ is the major component of the gas mixture.

The weak variation of the selectivity with pressure also suggests that the preference for CO₂ does not change significantly between 0.1 and 3 bar.


## Question 7 — Effect of gas composition and temperature

The CO₂ mole fraction is now decreased from 0.40 to 0.20, while the CH₄ mole fraction becomes 0.80. The IAST calculations are repeated at the same pressures to determine whether this change in gas composition affects the CO₂/CH₄ selectivity.


In [ ]:
compositions = {
    "yCO2 = 0.20": np.array([0.20, 0.80]),
    "yCO2 = 0.40": np.array([0.40, 0.60]),
}

selectivity_results = {}

for label, y_test in compositions.items():

    values = []

    for P in selectivity_pressures:

        q = pyiast.iast(
            P * y_test,
            [co2_isotherm, ch4_isotherm]
        )

        q_CO2 = q[0]
        q_CH4 = q[1]

        S = (q_CO2 / q_CH4) * (y_test[1] / y_test[0])

        values.append(S)

    selectivity_results[label] = values

In [ ]:
comparison_df = pd.DataFrame({
    "Pressure (bar)": selectivity_pressures,
    "Selectivity (yCO2 = 0.20)": selectivity_results["yCO2 = 0.20"],
    "Selectivity (yCO2 = 0.40)": selectivity_results["yCO2 = 0.40"]
})

comparison_df

### Effect of gas composition

Decreasing the CO₂ mole fraction from 0.40 to 0.20 gives very similar CO₂/CH₄ selectivity values over the studied pressure range.

The selectivity is therefore only weakly affected by this change in gas-phase composition.


### Effect of temperature

To determine whether the CO₂/CH₄ selectivity depends on temperature, pure-component CO₂ and CH₄ adsorption isotherms should be calculated at several temperatures.

For each temperature, the IAST calculation should then be repeated at the same pressure and gas composition. Comparing the resulting selectivities would show how temperature affects the adsorption preference of IRMOF-1.


## Question 8 — Evaluation of IRMOF-1 for CH₄ storage

IRMOF-1 has structural properties that are favorable for gas adsorption, with an accessible porosity of approximately **77.5%** and an accessible surface area of approximately **3965 Å²**.

However, the CH₄ adsorption isotherm reaches only approximately **1.13 mmol/g at 10.2 bar and 300 K**, corresponding to about **18 mg of CH₄ per gram of material**, or approximately **1.8 wt%**. Therefore, despite its high porosity, the amount of methane stored within the investigated pressure range remains relatively limited.

Based on these results, **IRMOF-1 does not appear to be a particularly good candidate for CH₄ storage under the conditions studied**. A highly porous structure alone is not sufficient: the material must also interact strongly enough with CH₄ to achieve a high adsorption capacity.

The isotherm is still increasing at 10.2 bar, so the maximum storage capacity has not been reached. Higher-pressure simulations would be necessary to characterize the complete adsorption behavior, but the present results do not provide evidence of a high CH₄ storage capacity.

For a practical storage application, the **deliverable capacity** should also be considered:

$$
q_{\mathrm{deliverable}}
=
q(P_{\mathrm{charge}})
-
q(P_{\mathrm{discharge}})
$$

because it represents the amount of stored methane that can actually be released when the pressure is decreased.


### Further explanation — Why does IRMOF-1 adsorb CO₂ more strongly than CH₄?

Although IRMOF-1 is highly porous and can adsorb both gases, the interaction between the gas molecules and the framework is different.

CO₂ has a significant **quadrupole moment**, which allows electrostatic interactions with the partial charges present in the IRMOF-1 framework, particularly around the Zn–O regions. CH₄ is non-polar and interacts mainly through weaker dispersion forces.

As a result, CO₂ has a stronger affinity for the framework than CH₄. This is consistent with the larger Henry coefficient, higher adsorption loading and CO₂/CH₄ selectivity obtained in the previous questions.
